# Customer Churn – Análisis Exploratorio y Preparación de Datos

**Notebook 01 · Fases 1 a 7 del proyecto**

> *Material académico*. Cada sección explica **qué** se hace y **por qué** se hace así,
> para que un estudiante pueda reproducir y comprender el flujo de un proyecto de
> ciencia de datos real, no solo ejecutar código.

## Contenido

| Fase | Sección |
|------|---------|
| 1 | Definición del problema, hipótesis y métricas |
| 2 | Carga del dataset |
| 3 | Auditoría de calidad |
| 4 | Limpieza y preparación (ETL) |
| 5 | Análisis Exploratorio (EDA) |
| 6 | Pruebas estadísticas |
| 7 | Correlaciones |

> Las fases 8–20 (selección de features, modelado, dashboard, reporte) se cubren en
> los notebooks `02_modeling.ipynb` y en la app Streamlit `app/dashboard.py`.


---
## Fase 1 · Definición del problema

### Contexto de negocio
Una empresa de telecomunicaciones (Telco) enfrenta un problema común en industrias
de suscripción: el **churn** (fuga de clientes). Retener un cliente es entre 5 y 25
veces más barato que adquirir uno nuevo (Harvard Business Review, 2014), por lo que
identificar a los clientes con mayor riesgo de irse permite actuar preventivamente.

### Preguntas a responder
1. **¿Qué tipo de clientes tienen más probabilidad de irse?**
2. **¿El contrato mensual vs anual afecta la retención?**
3. **¿Qué recomendaciones se pueden dar para reducir la fuga?**

### Hipótesis testeable
> **H1:** Los clientes con **contrato mes a mes** tienen una probabilidad de churn
> significativamente mayor que los clientes con contratos a uno o dos años.
> *(Esperamos confirmar con prueba chi-cuadrado y validar magnitud con tasa real.)*

> **H2:** La **antigüedad del cliente** (`tenure`) está negativamente correlacionada
> con la fuga: a mayor tiempo de permanencia, menor riesgo de churn.

> **H3:** Clientes que **no contratan servicios de soporte** (online security,
> tech support) tienen mayor churn por carecer de soporte cuando surgen problemas.

### Variable objetivo y predictoras
- **Objetivo:** `Churn` (categórica binaria: Yes / No → 1 / 0)
- **Predictoras:** 19 variables (demográficas, servicios contratados, facturación)
- **Tipo de problema:** Clasificación binaria supervisada

### Métricas de éxito
Dado que las clases están desbalanceadas (~26.5% churn) y la consecuencia de **no
detectar** un cliente que se va (falso negativo) es perder ingresos, priorizamos:

| Métrica | Por qué importa |
|---------|----------------|
| **Recall (clase 1)** | Captar la mayor cantidad de churners posibles |
| **F1-score (clase 1)** | Balance entre recall y precisión |
| **AUC-ROC** | Capacidad discriminativa global del modelo |
| **Accuracy** | Métrica auxiliar (engaña con clases desbalanceadas) |

### Restricciones
- Dataset estático (snapshot, no streaming).
- Sin información temporal del momento de la baja → no es serie de tiempo.
- 21 variables, 7,043 registros → entrenamiento factible en una laptop.


---
## Fase 2 · Carga del dataset

**Fuente:** [Telco Customer Churn – Kaggle (IBM)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

El archivo `CustomerChurn.csv` contiene 7,043 filas y 21 columnas. Lo cargamos
mediante el módulo `src/data_loader.py`, que además valida que estén presentes
todas las columnas esperadas (defensa contra archivos corruptos o cambios de esquema).


In [ ]:
# Imports y configuración inicial
import sys
sys.path.insert(0, '..')  # para poder importar el paquete `src`

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data, dataset_summary
from src.data_quality import (
    missing_values_report, duplicates_report,
    numeric_outliers_report, logical_range_checks,
)
from src.data_cleaning import clean_dataset, save_processed
from src.eda import (
    plot_target_distribution, plot_numeric_distributions,
    plot_categorical_vs_churn, plot_correlation_heatmap,
)
from src.stats_tests import (
    normality_test, levene_test, compare_groups,
    chi2_table, correlation_with_target,
)

# Semilla global para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# Cargamos el dataset crudo
df_raw = load_raw_data('../data/raw/CustomerChurn.csv')
df_raw.head()


In [ ]:
# Resumen estructural
summary = dataset_summary(df_raw)
print(f"Filas:    {summary['n_rows']:,}")
print(f"Columnas: {summary['n_cols']}")
print(f"Memoria:  {summary['memory_mb']} MB")
print("\nTipos de datos:")
df_raw.dtypes


**Observación importante:** `TotalCharges` aparece como **string (object)**, no
como número. Esto es una alerta: lo investigaremos en la auditoría de calidad.


---
## Fase 3 · Auditoría de calidad

Antes de transformar nada, **medimos**. Esta fase produce un reporte cuantitativo
que justifica las decisiones de limpieza posteriores. Tres ejes:

1. **Valores faltantes** (NaN explícitos *y* strings en blanco implícitos).
2. **Duplicados** (filas completas y IDs repetidos).
3. **Outliers** en variables numéricas (método IQR de Tukey).


In [ ]:
# 3.1 Valores faltantes
mv_report = missing_values_report(df_raw)
mv_report[mv_report['total_missing'] > 0]


**Hallazgo:** 11 filas (0.16%) tienen `TotalCharges` como string vacío
(`" "`). No son NaN convencionales, por eso pandas los cargó como object.
Investiguemos a qué clientes corresponden:


In [ ]:
# ¿Quiénes son esos 11 clientes con TotalCharges en blanco?
blank_mask = df_raw['TotalCharges'].astype(str).str.strip() == ''
df_raw.loc[blank_mask, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]


**Diagnóstico claro:** los 11 casos tienen `tenure == 0`, es decir, son
**clientes recién contratados que aún no han recibido su primera factura**. Por eso
el campo `TotalCharges` está vacío. Tiene sentido de negocio.

**Decisión de imputación:** asignar `TotalCharges = 0.0` (no han pagado nada todavía).
Alternativa descartada: eliminar las filas. La descartamos porque <0.2% es despreciable
y eliminar a los clientes nuevos sesgaría la muestra justo en el segmento más vulnerable.


In [ ]:
# 3.2 Duplicados
dup_report = duplicates_report(df_raw, id_col='customerID')
dup_report


Cero duplicados, perfecto. Cada `customerID` es único y no hay filas repetidas.


In [ ]:
# 3.3 Outliers en variables numéricas (método IQR de Tukey, k=1.5)
numeric_outliers_report(df_raw, ['tenure', 'MonthlyCharges'])


**Hallazgo:** **cero outliers** en `tenure` y `MonthlyCharges`. Sus rangos son
lógicos y acotados (tenure ∈ [0, 72], MonthlyCharges ∈ [18.25, 118.75]). No requiere
tratamiento. Para `TotalCharges` esperamos hasta haberla convertido a numérica.


In [ ]:
# 3.4 Validación de rangos lógicos
logical_range_checks(df_raw)


### Resumen de la auditoría

| Item | Resultado | Acción |
|------|-----------|--------|
| Valores faltantes | 11 blanks en `TotalCharges` (0.16%) | Imputar con 0.0 |
| Duplicados | 0 filas, 0 IDs | Ninguna |
| Outliers IQR | 0 en tenure y MonthlyCharges | Ninguna |
| Rangos lógicos | Todos válidos | Ninguna |
| Tipos de datos | `TotalCharges` viene como string | Convertir a float |
| `SeniorCitizen` | viene como 0/1 entero | Mapear a Yes/No (consistencia) |


---
## Fase 4 · Limpieza y preparación (ETL)

Aplicamos las decisiones de la auditoría a través del módulo `src/data_cleaning.py`.
Cada transformación está documentada en su docstring. Aquí ejecutamos y verificamos.

**Transformaciones aplicadas:**
1. `TotalCharges` → float, con imputación de blancos a 0.0.
2. `SeniorCitizen` → Yes/No (consistencia con el resto de variables sí/no).
3. `Churn` → 1/0 (target numérico).
4. Nombres de columnas → snake_case (estándar PEP 8).
5. **Variable derivada:** `tenure_group` (binning en cuatro tramos: 0-12, 13-24, 25-48, 49-72 meses).


In [ ]:
df = clean_dataset(df_raw)

# Verificamos: ¿queda algún faltante?
print(f"Total de NaN tras limpieza: {df.isna().sum().sum()}")
print(f"Total de filas: {len(df):,}")
print(f"Tasa global de churn: {df['churn'].mean()*100:.2f}%")

df.dtypes


In [ ]:
# Vista del dataset limpio
df.head()


In [ ]:
# Guardar versión procesada para que el dashboard la consuma directamente
save_processed(df, '../data/processed/churn_clean.csv')
print('✓ Dataset limpio guardado en data/processed/churn_clean.csv')


---
## Fase 5 · Análisis Exploratorio de Datos (EDA)

El EDA tiene tres niveles:

- **5.1 Variable objetivo:** ¿qué tan balanceadas están las clases?
- **5.2 Univariado:** distribuciones de cada variable.
- **5.3 Bivariado:** relación de cada variable con `churn`.

### 5.1 Distribución del target


In [ ]:
fig = plot_target_distribution(df)
fig.savefig('../reports/figures/01_target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


**Hallazgo:** clases desbalanceadas (~73.5% no-churn vs 26.5% churn). Esto tiene
dos implicaciones importantes para el modelado:

1. La **accuracy** será una métrica engañosa: un modelo que diga "todos se quedan"
   acierta el 73.5% sin esfuerzo.
2. En la división train/test usaremos **estratificación** y, en el entrenamiento,
   métricas robustas al desbalance (F1, recall, AUC-ROC).

### 5.2 Distribuciones univariadas (numéricas) por grupo de churn


In [ ]:
fig = plot_numeric_distributions(df, ['tenure', 'monthly_charges', 'total_charges'])
fig.savefig('../reports/figures/02_numeric_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


**Hallazgos visuales:**

- **`tenure`**: los churners se concentran fuertemente en los primeros meses;
  los no-churners están distribuidos más uniformemente y se acumulan en valores altos.
- **`monthly_charges`**: los churners pagan **más** mensualmente (~$74 vs ~$61).
- **`total_charges`**: los churners tienen menor acumulado histórico (lógico: se van
  pronto), aunque la distribución es muy sesgada por la relación tenure × charges.

### 5.3 Tasa de churn por variables categóricas


In [ ]:
cat_cols = ['contract', 'internet_service', 'payment_method',
            'online_security', 'tech_support', 'paperless_billing',
            'senior_citizen', 'partner', 'tenure_group']
fig = plot_categorical_vs_churn(df, cat_cols, ncols=3)
fig.savefig('../reports/figures/03_churn_rate_by_category.png', dpi=120, bbox_inches='tight')
plt.show()


**Hallazgos clave (línea punteada = tasa global 26.5%):**

| Variable | Patrón |
|----------|--------|
| **Contract** | Mes-a-mes ~43% vs Dos años ~3% — **15× más churn** |
| **Internet Service** | Fibra óptica ~42% vs DSL ~19% vs Sin internet ~7% |
| **Payment Method** | Cheque electrónico ~45% vs métodos automáticos ~15% |
| **Online Security / Tech Support** | Sin servicio ~42% vs Con servicio ~15% |
| **Tenure Group** | 0-12m ~48% vs 49-72m ~7% |
| **Senior Citizen** | Sí ~42% vs No ~24% |

Estos patrones ya empiezan a responder la pregunta del usuario, pero antes de
afirmar nada **validamos estadísticamente** en la Fase 6.


---
## Fase 6 · Pruebas estadísticas

Los gráficos sugieren patrones, pero necesitamos **confirmar formalmente** si las
diferencias observadas son estadísticamente significativas. Usamos α = 0.05.

### 6.1 Normalidad (Shapiro-Wilk)

H₀: los datos provienen de una distribución normal. Si p < 0.05, rechazamos H₀.


In [ ]:
normality_results = []
for col in ['tenure', 'monthly_charges', 'total_charges']:
    r = normality_test(df[col])
    r['variable'] = col
    normality_results.append(r)
pd.DataFrame(normality_results).set_index('variable')[['shapiro_W', 'p_value', 'is_normal']]


**Conclusión:** **ninguna** variable numérica es normal (todas con p ≈ 0). Esto
tiene tres consecuencias prácticas:

1. Usaremos **Spearman** (no Pearson) para correlaciones.
2. Usaremos **Mann-Whitney U** además del t-test para comparar grupos.
3. Para modelos sensibles a la distribución (regresión logística estándar, LDA)
   consideraremos transformaciones, aunque modelos basados en árboles (Random Forest,
   XGBoost) son inmunes a este supuesto.

### 6.2 Homogeneidad de varianzas (Levene)

H₀: las varianzas de los dos grupos (churn vs no-churn) son iguales.


In [ ]:
levene_results = []
for col in ['tenure', 'monthly_charges', 'total_charges']:
    r = levene_test(df, col)
    r['variable'] = col
    levene_results.append(r)
pd.DataFrame(levene_results).set_index('variable')


**Conclusión:** las varianzas **no** son iguales. Por eso usaremos
**Welch's t-test** (no asume igualdad de varianzas) en lugar del t-test clásico.

### 6.3 Comparación de medias por grupo (Welch + Mann-Whitney)


In [ ]:
compare_results = []
for col in ['tenure', 'monthly_charges', 'total_charges']:
    r = compare_groups(df, col)
    r['variable'] = col
    compare_results.append(r)
pd.DataFrame(compare_results).set_index('variable')[
    ['mean_group0', 'mean_group1', 'welch_p', 'mannwhitney_p']
]


**Conclusión:** las tres variables numéricas presentan diferencias
**altamente significativas** entre churners y no-churners (p < 10⁻⁷⁰ en todos los
casos, tanto en Welch como Mann-Whitney). Las diferencias son **reales**, no producto del azar.

### 6.4 Asociación de variables categóricas con churn (Chi² + Cramér's V)

- **Chi-cuadrado** dice *si hay* asociación (p-valor).
- **Cramér's V** dice *qué tan fuerte* es esa asociación (0 = independencia, 1 = perfecta).

Escala de interpretación (Cohen):
- V ≈ 0.1 → efecto pequeño
- V ≈ 0.3 → efecto moderado
- V ≈ 0.5 → efecto grande


In [ ]:
cat_for_chi2 = [
    'gender', 'senior_citizen', 'partner', 'dependents', 'phone_service',
    'multiple_lines', 'internet_service', 'online_security', 'online_backup',
    'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
    'contract', 'paperless_billing', 'payment_method', 'tenure_group',
]
chi2_results = chi2_table(df, cat_for_chi2)
chi2_results[['chi2', 'p_value', 'cramers_v', 'significant']].round(4)


**Hallazgos clave del chi-cuadrado:**

- **`gender` y `phone_service` NO son significativas** (p > 0.05). Es decir, el
  género del cliente no influye en si se va o no — un buen indicio de **ausencia
  de sesgo demográfico** en el problema. Podemos descartarlas como predictoras.
- **`contract` lidera con V = 0.41 (efecto cercano a grande)** → confirma H1.
- **Servicios de soporte (`online_security`, `tech_support`, `online_backup`)
  tienen V ≈ 0.29–0.35** → confirma H3.
- **`payment_method` tiene V = 0.30** → un patrón relevante: los pagos
  automáticos retienen más que el cheque electrónico.


---
## Fase 7 · Análisis de correlaciones

### 7.1 Correlación numérica con el target

Usamos **Spearman** porque ninguna variable numérica es normal (probado en 6.1).
Spearman captura relaciones **monotónicas** (no solo lineales) y es robusto a outliers.


In [ ]:
correlation_with_target(df, ['tenure', 'monthly_charges', 'total_charges'])


**Interpretación:**
- `tenure` ↔ churn: **-0.37** (correlación negativa moderada). A más antigüedad,
  menos churn. **Confirma H2**.
- `total_charges` ↔ churn: **-0.23** (moderada-baja). Lógicamente correlacionado
  con tenure.
- `monthly_charges` ↔ churn: **+0.18** (baja-moderada). Cargos altos elevan el riesgo.

### 7.2 Heatmap de correlación entre variables numéricas


In [ ]:
fig = plot_correlation_heatmap(df, ['tenure', 'monthly_charges', 'total_charges', 'churn'])
fig.savefig('../reports/figures/04_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


**Alerta de multicolinealidad:** `tenure` y `total_charges` están altamente
correlacionadas (Pearson ≈ 0.83), lo cual es esperable (TotalCharges ≈ tenure ×
MonthlyCharges). Lo tendremos en cuenta en la Fase 8 (Feature Selection) para evitar
incluir ambas en modelos lineales como regresión logística — podríamos eliminar una
o aplicar regularización L1/L2.


---
## Conclusiones preliminares (antes del modelado)

A partir de las Fases 1-7 ya podemos esbozar respuestas a las preguntas del proyecto.
El modelado posterior (Fases 8-16) **cuantificará** estos hallazgos.

### ¿Qué tipo de clientes tienen más probabilidad de irse?

Perfil del cliente en alto riesgo de churn:

- 📅 **Contrato mes-a-mes** (43% churn vs 3% en dos años).
- ⏱ **Antigüedad menor a 12 meses** (~48% churn).
- 🌐 **Internet de fibra óptica** (~42% churn — sorprendente; investigar en modelado).
- 💳 **Paga con cheque electrónico** (~45% churn).
- 🛡 **Sin servicios de soporte** (online security, tech support).
- 👴 **Senior citizen** (~42% vs 24% en no-seniors).
- 💰 **Cargos mensuales altos**.

### ¿El contrato mensual vs anual afecta la retención?

**Sí, con un impacto enorme.** Es la variable con **mayor asociación** al churn
(Cramér's V = 0.41, p ≈ 0). Diferencia visual: 43% vs 3% de churn (factor 15×).
La hipótesis H1 queda **soportada** por las pruebas estadísticas.

### ¿Qué recomendaciones se pueden dar? (preliminar)

1. **Migrar clientes mes-a-mes a contratos anuales** con incentivos (descuentos,
   meses gratis). Si se logra mover incluso 10-15% al contrato anual, el churn
   global caería notoriamente.
2. **Programa de onboarding** en los primeros 12 meses (la ventana de mayor riesgo).
3. **Bundle de soporte gratuito** (tech support + online security) los primeros 6 meses.
4. **Promover métodos de pago automáticos** sobre el cheque electrónico.
5. **Auditar la experiencia del cliente de fibra óptica** (su churn es desproporcionado).

> Las recomendaciones se refinarán y cuantificarán con el modelo predictivo
> (Fases 11-16): qué clientes específicos atacar primero (mayor probabilidad ×
> mayor valor).


---
## Próximo paso → `02_modeling.ipynb`

En el siguiente notebook abordaremos:
- Fase 8: Selección de features (VIF, importancia por árbol).
- Fase 9: División train/test estratificada.
- Fase 10: Escalado y encoding.
- Fases 11-14: Entrenamiento, cross-validation, tuning, evaluación.
- Fase 15: Análisis de errores e interpretabilidad (feature importance, SHAP).
- Fase 16: Conclusiones definitivas y recomendaciones cuantificadas.
